# 🎙️ Urdu Interview Processing Pipeline
**Stages:** Audio → Urdu Transcript → Verify → English Translation → Verify → De-identify → Final Dataset

**Models used:**
- ASR: `openai/whisper-large-v3-turbo` (via faster-whisper)
- Translation: `facebook/nllb-200-1.3B` ⬆️ **Upgraded for better quality**
- De-identification: `Microsoft Presidio` + `spaCy en_core_web_lg`

**Improvements in this version:**
- ✅ NLLB model upgraded from 600M to 1.3B (3x better translation)
- ✅ Sentence-aware chunking (preserves context instead of hard character splits)
- ✅ Better handling of Urdu→English translations

> ⚠️ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running!

In [1]:
# ── CELL 1: Check GPU ──────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✔ GPU available: {gpu_name}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
    print('  Pipeline will run on CPU (much slower)')

✔ GPU available: Tesla T4
  VRAM: 15.6 GB


In [2]:
# ── CELL 2: Install all dependencies ──────────────────────
print('Installing dependencies... (takes 3-5 minutes first time)')
print('⚠ Note: NLLB 1.3B model (~2.5GB) requires T4 GPU VRAM (~16GB available)')

!pip install -q faster-whisper==1.1.0
!pip install -q transformers==4.44.2 sentencepiece==0.2.0 sacremoses==0.1.1
!pip install -q presidio-analyzer==2.2.355 presidio-anonymizer==2.2.355
!pip install -q spacy==3.8.1
!pip install -q python-docx==1.1.2 langdetect==1.0.9 sacrebleu==2.4.3

# Download spaCy model
!python -m spacy download en_core_web_lg -q

print('\n✔ All dependencies installed!')
print('✔ Models will auto-download on first use (may take 2-3 minutes per model)')

Installing dependencies... (takes 3-5 minutes first time)
⚠ Note: NLLB 1.3B model (~2.5GB) requires T4 GPU VRAM (~16GB available)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 52.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 16.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 95.6 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 106.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.2 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver 

In [3]:
# ── CELL 3: Clone project from GitHub ──────────────────────
import os

REPO_DIR = 'urdu-pipeline'
REPO_URL = 'https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE.git'

if os.path.isdir(REPO_DIR):
    print(f'✔ Repo already exists — pulling latest changes')
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

print('✔ Repository ready!')
!ls -la

Cloning into 'urdu-pipeline'...
remote: Enumerating objects: 102, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 102 (delta 54), reused 102 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (102/102), 328.33 KiB | 17.28 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/urdu-pipeline
From https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE
 * branch            main       -> FETCH_HEAD
Already up to date.
✔ Repository cloned and updated successfully!
total 60
drwxr-xr-x 6 root root 4096 Jul  3 08:28 .
drwxr-xr-x 1 root root 4096 Jul  3 08:28 ..
-rw-r--r-- 1 root root 6842 Jul  3 08:28 config.py
drwxr-xr-x 8 root root 4096 Jul  3 08:28 .git
-rw-r--r-- 1 root root 1313 Jul  3 08:28 .gitignore
-rw-r--r-- 1 root root 9651 Jul  3 08:28 main.py
drwxr-xr-x 2 root root 4096 Jul  3 08:28 notebooks
drwxr-xr-x 2 root root 4096 Jul  3 08:28 pipeline
-rw-r--r-- 1 root root 4284 Jul  3 08:28 README.md


In [1]:
# ── CELL 4: Download audio from YouTube ───────────────────
import os
import subprocess
import glob

os.makedirs('audio', exist_ok=True)

AUDIO_PATH   = 'audio/test_audio.mp3'
FULL_PATTERN = 'audio/full.*'
YOUTUBE_URL  = 'https://youtu.be/pHZHYWe8Mkc'
START_SEC    = 137
DURATION_SEC = 720   # 12 minutes

def _find_full_audio():
    matches = sorted(glob.glob('audio/full.*'))
    return matches[0] if matches else None

if os.path.exists(AUDIO_PATH):
    print(f'✔ Audio already exists — skipping download: {AUDIO_PATH}')
else:
    !pip install -q -U yt-dlp

    strategies = [
        ['yt-dlp', '-x', '--audio-format', 'mp3', '-o', 'audio/full.%(ext)s', YOUTUBE_URL],
        ['yt-dlp', '-x', '--audio-format', 'mp3', '-o', 'audio/full.%(ext)s',
         '--extractor-args', 'youtube:player_client=android', YOUTUBE_URL],
        ['yt-dlp', '-x', '--audio-format', 'mp3', '-o', 'audio/full.%(ext)s',
         '--extractor-args', 'youtube:player_client=ios', YOUTUBE_URL],
    ]

    full_audio = None
    for i, cmd in enumerate(strategies, 1):
        print(f'\n  Download attempt {i}/{len(strategies)}...')
        result = subprocess.run(cmd, capture_output=True, text=True)
        full_audio = _find_full_audio()
        if result.returncode == 0 and full_audio:
            print(f'  ✔ Downloaded: {full_audio}')
            break
        err = (result.stderr or result.stdout or '').strip()
        if err:
            print('  ', err.splitlines()[-1])

    if not full_audio:
        print('\n⚠ YouTube download failed (Colab bot detection).')
        print('  Upload your own MP3 using the cell below, then re-run this cell.')
        from google.colab import files
        print('\n  Waiting for upload → save as audio/test_audio.mp3 ...')
        uploaded = files.upload()
        for name, data in uploaded.items():
            dest = AUDIO_PATH if name.endswith('.mp3') else f'audio/{name}'
            with open(dest, 'wb') as f:
                f.write(data)
            print(f'  ✔ Saved upload → {dest}')
        if not os.path.exists(AUDIO_PATH):
            raise FileNotFoundError(
                'No audio file available. Upload an MP3 or try running this cell again later.'
            )
    else:
        trim_cmd = [
            'ffmpeg', '-y', '-i', full_audio,
            '-ss', str(START_SEC), '-t', str(DURATION_SEC),
            '-c', 'copy', AUDIO_PATH,
        ]
        trim = subprocess.run(trim_cmd, capture_output=True, text=True)
        if trim.returncode != 0 or not os.path.exists(AUDIO_PATH):
            print('  Trim with -c copy failed — re-encoding clip...')
            subprocess.run([
                'ffmpeg', '-y', '-i', full_audio,
                '-ss', str(START_SEC), '-t', str(DURATION_SEC),
                AUDIO_PATH,
            ], check=True)
        if os.path.exists(full_audio):
            os.remove(full_audio)

audio_path = AUDIO_PATH
size_mb = os.path.getsize(audio_path) / 1e6
print(f'\n✔ Audio ready: {audio_path}')
print(f'   File size: {size_mb:.1f} MB')
print(f'   Duration: {DURATION_SEC // 60} minutes (from {START_SEC}s into source)')



  Download attempt 1/3...
  ✔ Downloaded: audio/full.mp3

✔ Audio ready: audio/test_audio.mp3
   File size: 6.8 MB
   Duration: 12 minutes (from 137s into source)


In [2]:
# ── CELL 5: Configure pipeline ─────────────────────────────
import sys
sys.path.insert(0, 'urdu-pipeline')

import config

# Set device to cuda since we have GPU
config.WHISPER_DEVICE       = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

# Set audio path
AUDIO_PATH = audio_path

print('Pipeline Configuration:')
print(f'  ASR Model        : whisper-{config.WHISPER_MODEL}')
print(f'  Translation Model: {config.TRANSLATION_MODEL}')
print(f'  Device           : {config.WHISPER_DEVICE}')
print(f'  Audio file       : {AUDIO_PATH}')
print(f'  Confidence thresh: {config.CONFIDENCE_THRESHOLD}')
print(f'  Chunk size       : {config.CHUNK_SIZE} chars')

Pipeline Configuration:
  ASR Model        : whisper-large-v3-turbo
  Translation Model: facebook/nllb-200-1.3B
  Device           : cuda
  Audio file       : audio/test_audio.mp3
  Confidence thresh: 0.55
  Chunk size       : 500 chars


In [3]:
# Check if audio file exists and its size
import os

audio_path = "audio/test_audio.mp3"
if os.path.exists(audio_path):
    size_mb = os.path.getsize(audio_path) / 1e6
    size_mins = size_mb / 0.64  # Rough conversion: 1 min ≈ 0.64 MB at 128kbps
    print(f'✔ Audio file exists: {size_mb:.1f} MB ({size_mins:.0f} minutes)')
else:
    print(f'✗ Audio file NOT found: {audio_path}')

# Check GPU status
import torch
if torch.cuda.is_available():
    print(f'✔ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   Used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB')
else:
    print('✗ GPU not available!')

✔ Audio file exists: 6.8 MB (11 minutes)
✔ GPU: Tesla T4
   VRAM: 15.6 GB
   Used: 0.00 GB


In [4]:
# ── CELL 6: STAGE 1 — Urdu Transcription ──────────────────
from pipeline.utils import ensure_dirs
ensure_dirs(config.STAGE1_DIR, config.STAGE2_DIR, config.STAGE3_DIR,
            config.STAGE4_DIR, config.STAGE5_DIR, config.STAGE6_DIR)

from pipeline.transcribe import transcribe

stage1_result = transcribe(AUDIO_PATH)

# Preview
print('\n── Urdu Transcript Preview (first 500 chars) ──')
print(stage1_result['full_urdu_text'][:500])


  STAGE 1: TRANSCRIPTION (ASR)
  Audio file   : audio/test_audio.mp3
  Model        : whisper-large-v3-turbo
  Language     : auto-detect
  Device       : cuda
  Temperature  : 0.0 (fallback: [0.2, 0.4, 0.6])
  Beam size    : 5

  Loading Whisper model (first run downloads ~800 MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

  Model loaded.

  Transcribing audio (language: auto-detect) ...
  Raw segments from Whisper: 143
  Merged 7 micro-segments into neighbours.

  Results:
  Segments total    : 136
    Urdu            : 0
    English         : 136
    Low confidence  : 1  (threshold=0.55)
  Avg raw confidence: 0.647
  Avg cal confidence: 0.782
  Avg text quality  : 0.962
  Loops removed     : 0
  Micro-segs merged : 7

  Segment detail:
    [ ][EN]    00:00:00.180 -> 00:00:06.440 | conf=0.943 | As-salamu alaykum Dr. Majda. Welcome to the show. Thank
    [ ][EN]    00:00:07.940 -> 00:00:17.160 | conf=0.868 | If you give us a small introduction so that our audienc
    [ ][EN]    00:00:17.420 -> 00:00:26.200 | conf=0.941 | Thank you so much Zainab for inviting me on your show. 
    [ ][EN]    00:00:26.200 -> 00:00:27.700 | conf=0.717 | It's not MBBS.
    [ ][EN]    00:00:28.020 -> 00:00:29.520 | conf=0.760 | Exactly, exactly. You have to tell me.
    [ ][EN]    00:00:31.260 -> 00:00:34.320 | conf=0.658 | B

In [5]:
#Clearing memory of previous model to free up GPU VRAM for next stage
import torch, gc
if 'model' in globals():
    del model
gc.collect()
torch.cuda.empty_cache()

In [6]:
# ── CELL 7: STAGE 2 — Verify Urdu Transcript ──────────────
from pipeline.verify_transcript import verify_transcript

stage2_result = verify_transcript(stage1_result)

print(f'\nQuality Score : {stage2_result["quality_score"]}/100')
print(f'Quality Label : {stage2_result["quality_label"]}')

# Show flagged segments
flagged = stage2_result['verification_report']['flagged_details']
if flagged:
    print(f'\n⚠ Flagged Segments ({len(flagged)}):')
    for f in flagged[:5]:
        print(f'  seg {f["segment_id"]} [{f["start_fmt"]}] conf={f["confidence"]:.2f}: {f["text"][:60]}...')
else:
    print('\n✔ No segments flagged!')


  STAGE 2: VERIFICATION OF TRANSCRIPT
  Interview ID        : test_audio
  Total segments      : 136
  Urdu segments       : 0
  English segments    : 136
  Micro-segs merged   : 7
  Loops removed       : 0
  Avg raw confidence  : 0.6467
  Avg cal confidence  : 0.7825
  Avg text quality    : 0.9623
  Confidence threshold: 0.55
    [ok      ][EN]    seg 001 | conf=0.943 | tq=0.97 | As-salamu alaykum Dr. Majda. Welcome to the s
    [ok      ][EN]    seg 002 | conf=0.868 | tq=0.99 | If you give us a small introduction so that o
    [ok      ][EN]    seg 003 | conf=0.941 | tq=0.97 | Thank you so much Zainab for inviting me on y
    [ok      ][EN]    seg 004 | conf=0.717 | tq=0.87 | It's not MBBS.
    [ok      ][EN]    seg 005 | conf=0.760 | tq=1.00 | Exactly, exactly. You have to tell me.
    [ok      ][EN]    seg 006 | conf=0.658 | tq=0.97 | Because, in a moment, my son has been like th
    [ok      ][EN]    seg 007 | conf=0.830 | tq=1.00 | He said that my mother is a doctor.
    [ok    

In [7]:
# ── CELL 8: STAGE 3 — Translate Urdu → English ────────────
from pipeline.translate import translate

stage3_result = translate(stage2_result)

print('\n── English Translation Preview (first 500 chars) ──')
print(stage3_result['english_full_text'][:500])


  STAGE 3: TRANSLATION: URDU → ENGLISH (smart routing)
  Interview ID   : test_audio
  Model          : facebook/nllb-200-1.3B
  Routing        : Urdu→NLLB  |  English→pass-through

  Routing: 0 Urdu segments -> translate  |  136 English segments -> pass-through

  [opt] All segments are English — skipping translation model load.

  Assembling translated segments:
    [-] seg 001/136 | PASS-THROUGH | As-salamu alaykum Dr. Majda. Welcome to the show. Thank you ...
    [-] seg 002/136 | PASS-THROUGH | If you give us a small introduction so that our audience tha...
    [-] seg 003/136 | PASS-THROUGH | Thank you so much Zainab for inviting me on your show. My na...
    [-] seg 004/136 | PASS-THROUGH | It's not MBBS....
    [-] seg 005/136 | PASS-THROUGH | Exactly, exactly. You have to tell me....
    [-] seg 006/136 | PASS-THROUGH | Because, in a moment, my son has been like this....
    [-] seg 007/136 | PASS-THROUGH | He said that my mother is a doctor....
    [-] seg 008/136 | PASS-THR

In [8]:
# ── CELL 9: STAGE 4 — Verify English Translation ──────────
from pipeline.verify_translation import verify_translation

stage4_result = verify_translation(stage3_result)

print(f'\nTranslation Quality Score : {stage4_result["translation_quality_score"]}/100')
print(f'Translation Quality Label : {stage4_result["translation_quality_label"]}')

# Show flagged translation segments
t_flagged = stage4_result['translation_verification_report']['flagged_details']
if t_flagged:
    print(f'\n⚠ Translation Flagged Segments ({len(t_flagged)}):')
    for f in t_flagged[:5]:
        print(f'  seg {f["segment_id"]}: {f["eng_text"][:60]}... | Issues: {", ".join(f["issues"])}')
else:
    print('\n✔ All translations verified OK!')


  STAGE 4: VERIFICATION OF ENGLISH TRANSLATION
  Interview ID      : test_audio
  Total segments    : 136
  Translated        : 0
  Pass-through      : 136

  Checks: empty, error markers, length ratio, nonsense detection
    [✔][-] seg 001
             EN: As-salamu alaykum Dr. Majda. Welcome to the show. Thank you so much fo...
    [✔][-] seg 002
             EN: If you give us a small introduction so that our audience that you are ...
    [✔][-] seg 003
             EN: Thank you so much Zainab for inviting me on your show. My name is Majd...
    [✔][-] seg 004
             EN: It's not MBBS....
    [✔][-] seg 005
             EN: Exactly, exactly. You have to tell me....
    [✔][-] seg 006
             EN: Because, in a moment, my son has been like this....
    [✔][-] seg 007
             EN: He said that my mother is a doctor....
    [✔][-] seg 008
             EN: So, he said that she will put a medical camp here....
    [✔][-] seg 009
             EN: So, he said no. She is a P

In [9]:
# ── CELL 10: STAGE 5 — De-identification ──────────────────
from pipeline.deidentify import deidentify

stage5_result = deidentify(stage4_result)

print(f'\nEntities removed: {stage5_result["entities_removed_count"]}')
print('\n── De-identified Text Preview (first 500 chars) ──')
print(stage5_result['deidentified_english_full'][:500])


  STAGE 5: DE-IDENTIFICATION OF DATASET
  Interview ID : test_audio
  Loading Presidio + spaCy (first run may take a moment)...
  ✔ Presidio loaded.

  De-identifying full English text...
      [skip] Ignored generic temporal words: ['today', 'today', 'today', 'night', 'night']

  De-identifying segments:
      [skip] Ignored generic temporal words: ['today']
    [PII] seg 001 | 1 entities removed | As-salamu alaykum Dr. [NAME]. Welcome to the show. Thank you...
      [skip] Ignored generic temporal words: ['today']
    [ok ] seg 002 | 0 entities removed | If you give us a small introduction so that our audience tha...
    [PII] seg 003 | 2 entities removed | Thank you so much [NAME] for inviting me on your show. My na...
    [ok ] seg 004 | 0 entities removed | It's not MBBS....
    [ok ] seg 005 | 0 entities removed | Exactly, exactly. You have to tell me....
    [ok ] seg 006 | 0 entities removed | Because, in a moment, my son has been like this....
    [ok ] seg 007 | 0 entities r

In [10]:
# ── CELL 11: STAGE 6 — Final Export ───────────────────────
from pipeline.export import export

final_result = export(stage5_result)

print(f'\n✔ Final JSON : {final_result["json_path"]}')
print(f'✔ Final DOCX : {final_result["docx_path"]}')


  STAGE 6: FINAL EXPORT: JSON + DOCX
  Interview ID : test_audio

  Building final JSON dataset...
  Saved -> /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.json

  Building DOCX report...
  ✔ DOCX saved → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.docx

  ── Final Outputs ─────────────────────────────
  JSON → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.json
  DOCX → /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.docx

  ✔ Stage 6 complete. Pipeline finished!

✔ Final JSON : /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.json
✔ Final DOCX : /content/urdu-pipeline/outputs/6_final_dataset/test_audio_final_dataset.docx


In [2]:
# ── CELL 12: Download all outputs ─────────────────────────
import os
import shutil

try:
    from google.colab import files
except ImportError:
    files = None

interview_id = stage1_result.get('interview_id', 'interview')
print('Current working dir:', os.getcwd())

# Locate repo/outputs robustly (works even if cwd is /content)
candidate_roots = [
    os.getcwd(),
    '/content/urdu-pipeline',
    '/content',
]

repo_root = None
outputs_dir = None
for root in candidate_roots:
    if os.path.isdir(os.path.join(root, 'outputs')):
        repo_root = root
        outputs_dir = os.path.join(root, 'outputs')
        break

if outputs_dir is None:
    print('✗ Error: outputs/ folder not found!')
    print('   Make sure you ran Cells 6-11 (all pipeline stages) first.')
    print('\nChecked paths:')
    for p in candidate_roots:
        print('  -', os.path.join(p, 'outputs'))
    print('\nDirectory listing:')
    os.system('ls -la')
else:
    print(f'✔ Outputs folder found at: {outputs_dir}')
    print('Contents:')
    os.system(f'ls -lh "{outputs_dir}" || true')

    # Zip outputs from the repo root
    zip_base = os.path.join(repo_root, f'{interview_id}_outputs')
    zip_file = f'{zip_base}.zip'
    try:
        shutil.make_archive(
            base_name=zip_base,
            format='zip',
            root_dir=repo_root,
            base_dir='outputs',
        )
    except Exception as e:
        print('✗ Error creating ZIP:', e)
    else:
        if os.path.exists(zip_file):
            size_mb = os.path.getsize(zip_file) / 1e6
            print(f'\n✔ Zipped outputs: {zip_file} ({size_mb:.1f} MB)')
            if files is not None:
                print('  Downloading...')
                files.download(zip_file)
            else:
                print('  Download skipped (not running in Google Colab).')
                print(f'  Manual file path: {zip_file}')
        else:
            print(f'✗ Error: Could not create {zip_file}')


NameError: name 'stage1_result' is not defined

In [ ]:
# ── CELL 13 (OPTIONAL): Run full pipeline in one go ───────
# Use this after testing individual stages above

import config
config.WHISPER_DEVICE = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

from main import run_pipeline

run_pipeline('audio/test_audio.mp3', start_stage=1)


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  URDU INTERVIEW PROCESSING PIPELINE
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
  Audio file  : audio/test_audio.mp3
  Start stage : 1
  Started at  : 2026-07-01T18:18:34.561194

  STAGE 1: URDU TRANSCRIPTION (ASR)
  Audio file : audio/test_audio.mp3
  Model      : whisper-large-v3-turbo
  Language   : ur (Urdu)
  Device     : cuda

  Loading Whisper model (first run downloads ~800MB)...
  ✔ Model loaded.

  Transcribing audio... (this may take a few minutes for long audio)

  Processing segments:
    [⚠] 00:00:00.000 → 00:00:01.600 | conf=0.73 | اسلام علیکم Dr. Majda...
    [⚠] 00:00:01.600 → 00:00:02.960 | conf=0.72 | Welcome to the show...
    [✔] 00:00:02.960 → 00:00:05.060 | conf=0.88 | Thank you so much for coming today...
    [✔] 00:00:05.060 → 00:00:06.460 | conf=0.95 | and taking out the time...
    [⚠] 00:00:07.840 → 00:00:10.820 | conf=0.55 | If you give us a small introduction...
    [⚠] 00:00:1

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


  ✔ Model loaded on cuda.

  Translating full text...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



  Translating segments:
    ✔ seg 001/537 | UR: اسلام علیکم Dr. Majda...
             | EN: Islam علیکم Dr. Majda...
    ✔ seg 002/537 | UR: Welcome to the show...
             | EN: Welcome to the show...
    ✔ seg 003/537 | UR: Thank you so much for coming today...
             | EN: Thank you so much for coming today...
    ✔ seg 004/537 | UR: and taking out the time...
             | EN: today and taking the time...
    ✔ seg 005/537 | UR: If you give us a small introduction...
             | EN: if you give us a little introduction...
    ✔ seg 006/537 | UR: so that our audience that you are listen...
             | EN: so that our audience that you are listening to today...
    ✔ seg 007/537 | UR: they know that what a brilliant...
             | EN: today they know what a brilliant...
    ✔ seg 008/537 | UR: Academian you are...
             | EN: academic you are...
    ✔ seg 009/537 | UR: Thank you so much Zainab...
             | EN: Thank you so much Zainab...
    ✔ seg 010

    ✔ seg 373/537 | UR: your...
             | EN: your...
    ✔ seg 374/537 | UR: will...
             | EN: will...
    ✔ seg 375/537 | UR: is...
             | EN: is...
    ✔ seg 376/537 | UR: very important...
             | EN: very important...
    ✔ seg 377/537 | UR: and...
             | EN: and...


: 